# SVC & XGBoost

## Cargamos las representaciones y entrenamos los modelos

Entrenamos siempre con todo train y evaluamos en validation, sin cross-validation.
Cada modelo se evalúa de dos formas: con todo validation y excluyendo solo de validation
los prompts que contienen [PROMPT] o [TARGETLANGUAGE].

Optuna ajusta los hiperparámetros usando todo validation y después repetimos ambas
evaluaciones. Elegimos el mejor modelo por MCC en validation completo y usamos test
únicamente para su evaluación final. Reutilizamos las matrices `.npz` del notebook 02,
cuyos vectorizadores se ajustaron solo con train. El tuning modifica solo los clasificadores;
`min_df` y los n-gramas quedan fijados por esas representaciones.

## TF-iDF


In [7]:
import pandas as pd
from scipy.sparse import load_npz

train = pd.read_json("../data/interim/train_tok.jsonl", lines=True)
y_train = train["jailbreak"]

val = pd.read_json("../data/interim/val_tok.jsonl", lines=True)
y_val = val["jailbreak"]
# Misma detección de marcadores que en el notebook 02; no se filtra train.  
val["has_prompt_marker"] = val["prompt_clean"].str.contains(
    r"\[\s*(?:prompt|targetlanguage)\s*\]", case=False, regex=True, na=False
)
validation_masks = {
    "Todos": pd.Series(True, index=val.index),
    "Sin marcadores": ~val["has_prompt_marker"],
}

# Los JSONL aportan las etiquetas y los marcadores; las features vienen de las matrices.
X_train_tfidf = load_npz("../data/processed/tfidf_train.npz")
X_val_tfidf = load_npz("../data/processed/tfidf_val.npz")
X_train, X_val = X_train_tfidf, X_val_tfidf
assert X_train.shape[0] == len(y_train) and X_val.shape[0] == len(y_val)
assert X_train.shape[1] == X_val.shape[1]

In [8]:
from sklearn.svm import LinearSVC
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef)


def evaluate(y_true, y_pred):
    return {
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


trained_models = {}

# 1. Definir los modelos
models = {
    "LinearSVC": LinearSVC(class_weight="balanced", max_iter=10000),
    "XGBoost": XGBClassifier(random_state=42, tree_method="hist", n_jobs=-1),
}

# 2. Entrenar con todo train y evaluar de las dos formas en validation
results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    trained_models[("TF-IDF", name, "Inicial")] = pipe
    y_pred = pipe.predict(X_val)
    for variant, mask in validation_masks.items():
        results.append({
            "Modelo": name,
            "Validation": variant,
            **evaluate(y_val[mask], y_pred[mask]),
        })

# 3. Comparar resultados
df_results = pd.DataFrame(results)
print(df_results.sort_values("MCC", ascending=False))

# Guardamos una copia para la tabla comparativa final
df_results_tfidf = df_results.copy()

      Modelo      Validation        F1  Accuracy  Precision    Recall  \
0  LinearSVC           Todos  0.568807  0.930025   0.574074  0.563636   
1  LinearSVC  Sin marcadores  0.573209  0.909750   0.585987  0.560976   
2    XGBoost           Todos  0.527473  0.935980   0.666667  0.436364   
3    XGBoost  Sin marcadores  0.535316  0.917655   0.685714  0.439024   

        MCC  
0  0.530760  
1  0.522931  
2  0.507548  
3  0.507282  


In [9]:
import optuna
from sklearn.svm import LinearSVC
import json


def objective(trial, model_name):
    if model_name == "LinearSVC":
        # Hiperparámetros propuestos por Optuna
        C = trial.suggest_float("C", 1e-3, 100, log=True)

        # Nuevos hiperparámetros de LinearSVC
        penalty = trial.suggest_categorical(
            "penalty", ["l1", "l2"]
        )

        class_weight = trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        )

        # Evitar combinaciones incompatibles
        if penalty == "l1":
            loss = "squared_hinge"
            dual = False
        else:
            loss = trial.suggest_categorical(
                "loss", ["hinge", "squared_hinge"]
            )
            dual = True if loss == "hinge" else "auto"

        clf = LinearSVC(
            C=C, penalty=penalty, loss=loss, dual=dual,
            class_weight=class_weight, random_state=42, max_iter=10000
        )
    else:
        clf = XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            scale_pos_weight=trial.suggest_categorical(
                "scale_pos_weight", [1.0, float(y_train.eq(0).sum() / y_train.eq(1).sum())]
            ),
            random_state=42, tree_method="hist", n_jobs=-1,
        )

    clf.fit(X_train, y_train)
    return matthews_corrcoef(y_val, clf.predict(X_val))


for model_name, prefix in [("LinearSVC", "svc"), ("XGBoost", "xgb")]:
    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(lambda trial: objective(trial, model_name), n_trials=25)
    print(model_name, "Mejor MCC en validation:", study.best_value)
    print("Mejores parámetros:", study.best_params)
    # Archivos distintos de los antiguos resultados de cross-validation.
    with open(f"../data/processed/{prefix}_tfidf_validation_best_params.json", "w") as f:
        json.dump(study.best_params, f, indent=2)

[I 2026-11-11 18:26:24,139] A new study created in memory with name: no-name-13c0fb47-da8a-44be-8bce-e36f37f26fce
[I 2026-11-11 18:26:24,286] Trial 0 finished with value: 0.4054024871789971 and parameters: {'C': 0.0745934328572655, 'penalty': 'l1', 'class_weight': None}. Best is trial 0 with value: 0.4054024871789971.
[I 2026-11-11 18:26:24,360] Trial 1 finished with value: 0.4587732058743649 and parameters: {'C': 0.0060252157362038605, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'squared_hinge'}. Best is trial 1 with value: 0.4587732058743649.
c:\Users\iker\Documents\uni\4\1_semestre\nlp\proyecto\jailbreak-prompt-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[I 2026-11-11 18:26:56,585] Trial 2 finished with value: 0.45909906155712166 and parameters: {'C': 14.528246637516036, 'penalty': 'l1', 'class_weight': 'balanced'}. Best is trial 2 with value: 0.45909906155

LinearSVC Mejor MCC en validation: 0.5527415371519223
Mejores parámetros: {'C': 3.2965837591261384, 'penalty': 'l2', 'class_weight': None, 'loss': 'hinge'}


[I 2026-11-11 18:29:33,756] Trial 0 finished with value: 0.5080975523677368 and parameters: {'n_estimators': 250, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.5779972601681014, 'scale_pos_weight': 10.259604190919674}. Best is trial 0 with value: 0.5080975523677368.
[I 2026-11-11 18:30:31,584] Trial 1 finished with value: 0.4973363958379586 and parameters: {'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.010725209743171996, 'min_child_weight': 10, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6061695553391381, 'scale_pos_weight': 10.259604190919674}. Best is trial 0 with value: 0.5080975523677368.
[I 2026-11-11 18:30:57,312] Trial 2 finished with value: 0.5175784380096994 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.04345454109729477, 'min_child_weight': 3, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.569746930326021, 'scale_pos_weight': 10.2596041

XGBoost Mejor MCC en validation: 0.5363093943250722
Mejores parámetros: {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.048340078586424924, 'min_child_weight': 10, 'subsample': 0.9577484666365966, 'colsample_bytree': 0.8029445582556498, 'scale_pos_weight': 10.259604190919674}


## BoW


In [10]:
# Mismas filas y etiquetas que TF-IDF, con la representación BoW ya calculada.
X_train_bow = load_npz("../data/processed/bow_train.npz")
X_val_bow = load_npz("../data/processed/bow_val.npz")
X_train, X_val = X_train_bow, X_val_bow

In [11]:
# Las filas conservan el orden de los JSONL con los que se generaron en el notebook 02.
assert X_train.shape[0] == len(y_train) and X_val.shape[0] == len(y_val)
assert X_train.shape[1] == X_val.shape[1]

In [24]:
from sklearn.svm import LinearSVC
import pandas as pd

# 1. Definir los modelos
models = {
    "LinearSVC": LinearSVC(class_weight="balanced", max_iter=50000),
    "XGBoost": XGBClassifier(random_state=42, tree_method="hist", n_jobs=-1),
}

# 2. Entrenar con todo train y evaluar de las dos formas en validation
results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    trained_models[("BoW", name, "Inicial")] = pipe
    y_pred = pipe.predict(X_val)
    for variant, mask in validation_masks.items():
        results.append({
            "Modelo": name,
            "Validation": variant,
            **evaluate(y_val[mask], y_pred[mask]),
        })

# 3. Comparar resultados
df_results = pd.DataFrame(results)
print(df_results.sort_values("MCC", ascending=False))

# Guardamos una copia para la tabla comparativa final
df_results_bow = df_results.copy()

      Modelo      Validation        F1  Accuracy  Precision    Recall  \
2    XGBoost           Todos  0.543478  0.937469   0.675676  0.454545   
3    XGBoost  Sin marcadores  0.551471  0.919631   0.694444  0.457317   
0  LinearSVC           Todos  0.426901  0.902730   0.412429  0.442424   
1  LinearSVC  Sin marcadores  0.439759  0.877470   0.434524  0.445122   

        MCC  
2  0.522887  
3  0.522808  
0  0.374102  
1  0.371017  


In [18]:
import optuna
from sklearn.svm import LinearSVC
import json


def objective(trial, model_name):
    if model_name == "LinearSVC":
        # Hiperparámetros propuestos por Optuna
        C = trial.suggest_float("C", 1e-3, 100, log=True)

        # Nuevos hiperparámetros de LinearSVC
        penalty = trial.suggest_categorical(
            "penalty", ["l1", "l2"]
        )

        class_weight = trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        )

        # Evitar combinaciones incompatibles
        if penalty == "l1":
            loss = "squared_hinge"
            dual = False
        else:
            loss = trial.suggest_categorical(
                "loss", ["hinge", "squared_hinge"]
            )
            dual = True if loss == "hinge" else "auto"

        clf = LinearSVC(
            C=C, penalty=penalty, loss=loss, dual=dual,
            class_weight=class_weight, random_state=42, max_iter=50000
        )
    else:
        clf = XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            scale_pos_weight=trial.suggest_categorical(
                "scale_pos_weight", [1.0, float(y_train.eq(0).sum() / y_train.eq(1).sum())]
            ),
            random_state=42, tree_method="hist", n_jobs=-1,
        )

    clf.fit(X_train, y_train)
    return matthews_corrcoef(y_val, clf.predict(X_val))


for model_name, prefix in [("LinearSVC", "svc"), ("XGBoost", "xgb")]:
    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(lambda trial: objective(trial, model_name), n_trials=25)
    print(model_name, "Mejor MCC en validation:", study.best_value)
    print("Mejores parámetros:", study.best_params)
    # Archivos distintos de los antiguos resultados de cross-validation.
    with open(f"../data/processed/{prefix}_bow_validation_best_params.json", "w") as f:
        json.dump(study.best_params, f, indent=2)

[I 2026-11-11 19:34:35,135] A new study created in memory with name: no-name-53949f42-6e31-496d-8e29-82cf776e7601
[I 2026-11-11 19:34:42,643] Trial 0 finished with value: 0.4700826080365764 and parameters: {'C': 0.0745934328572655, 'penalty': 'l1', 'class_weight': None}. Best is trial 0 with value: 0.4700826080365764.
[I 2026-11-11 19:35:14,160] Trial 1 finished with value: 0.5133731042712649 and parameters: {'C': 0.0060252157362038605, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'squared_hinge'}. Best is trial 1 with value: 0.5133731042712649.
c:\Users\iker\Documents\uni\4\1_semestre\nlp\proyecto\jailbreak-prompt-detection\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[I 2026-11-11 19:38:05,118] Trial 2 finished with value: 0.36401574400903886 and parameters: {'C': 14.528246637516036, 'penalty': 'l1', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.51337310427

LinearSVC Mejor MCC en validation: 0.5229930074171107
Mejores parámetros: {'C': 0.007646654016389435, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'hinge'}


[I 2026-11-11 19:51:05,477] Trial 0 finished with value: 0.556529244697473 and parameters: {'n_estimators': 250, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.5779972601681014, 'scale_pos_weight': 10.259604190919674}. Best is trial 0 with value: 0.556529244697473.
[I 2026-11-11 19:51:14,045] Trial 1 finished with value: 0.5034817337900015 and parameters: {'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.010725209743171996, 'min_child_weight': 10, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6061695553391381, 'scale_pos_weight': 10.259604190919674}. Best is trial 0 with value: 0.556529244697473.
[I 2026-11-11 19:51:17,936] Trial 2 finished with value: 0.5240862072317288 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.04345454109729477, 'min_child_weight': 3, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.569746930326021, 'scale_pos_weight': 10.2596041909

XGBoost Mejor MCC en validation: 0.5605622167774889
Mejores parámetros: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.1085423741663872, 'min_child_weight': 6, 'subsample': 0.6340144604952528, 'colsample_bytree': 0.7298923571559777, 'scale_pos_weight': 10.259604190919674}


## Comparing models


In [25]:
# Tabla comparativa en validation con hiperparámetros iniciales (sin Optuna)
cols = ["Representación", "Modelo", "Validation", "F1", "Accuracy", "Precision", "Recall", "MCC"]

df_compare = pd.concat(
    [
        df_results_tfidf.assign(Representación="TF-IDF"),
        df_results_bow.assign(Representación="BoW"),
    ],
    ignore_index=True,
)[cols]

df_compare = df_compare.sort_values("MCC", ascending=False).reset_index(drop=True)
df_compare.round(4)

,Representación,Modelo,Validation,F1,Accuracy,Precision,Recall,MCC
0,TF-IDF,LinearSVC,Todos,0.5688,0.9300,0.5741,0.5636,0.5308
1,TF-IDF,LinearSVC,Sin marcadores,0.5732,0.9097,0.5860,0.5610,0.5229
2,BoW,XGBoost,Todos,0.5435,0.9375,0.6757,0.4545,0.5229
3,BoW,XGBoost,Sin marcadores,0.5515,0.9196,0.6944,0.4573,0.5228
4,TF-IDF,XGBoost,Todos,0.5275,0.9360,0.6667,0.4364,0.5075
5,TF-IDF,XGBoost,Sin marcadores,0.5353,0.9177,0.6857,0.4390,0.5073
6,BoW,LinearSVC,Todos,0.4269,0.9027,0.4124,0.4424,0.3741
7,BoW,LinearSVC,Sin marcadores,0.4398,0.8775,0.4345,0.4451,0.3710


### Resultados en validation con Optuna (SVM y XGBoost)

In [22]:
import json
from sklearn.svm import LinearSVC


def build_tuned_svc(params):
    """LinearSVC con los hiperparámetros guardados por Optuna (JSON).
    loss y dual no siempre están en el JSON (con penalty="l1" no se sugieren):
    se rederivan con la misma regla que en objective()."""
    if params["penalty"] == "l1":
        loss, dual = "squared_hinge", False
    else:
        loss = params["loss"]
        dual = True if loss == "hinge" else "auto"
    return LinearSVC(
        C=params["C"],
        penalty=params["penalty"],
        loss=loss,
        dual=dual,
        class_weight=params["class_weight"],
        random_state=42,
        max_iter=50000,
    )


# Mismos datos y métricas que en la tabla sin Optuna.
optuna_configs = {
    ("TF-IDF", "LinearSVC"): (X_train_tfidf, X_val_tfidf, "../data/processed/svc_tfidf_validation_best_params.json"),
    ("TF-IDF", "XGBoost"): (X_train_tfidf, X_val_tfidf, "../data/processed/xgb_tfidf_validation_best_params.json"),
    ("BoW", "LinearSVC"): (X_train_bow, X_val_bow, "../data/processed/svc_bow_validation_best_params.json"),
    ("BoW", "XGBoost"): (X_train_bow, X_val_bow, "../data/processed/xgb_bow_validation_best_params.json"),
}

rows = []
for (rep, name), (X_train, X_val, params_path) in optuna_configs.items():
    with open(params_path) as f:
        params = json.load(f)
    if name == "LinearSVC":
        pipe = build_tuned_svc(params)
    else:
        pipe = XGBClassifier(random_state=42, tree_method="hist", n_jobs=-1, **params)
    pipe.fit(X_train, y_train)
    trained_models[(rep, name, "Optuna")] = pipe
    y_pred = pipe.predict(X_val)
    for variant, mask in validation_masks.items():
        rows.append({
            "Representación": rep,
            "Modelo": name,
            "Validation": variant,
            **evaluate(y_val[mask], y_pred[mask]),
        })

df_compare_optuna = pd.DataFrame(rows).sort_values("MCC", ascending=False).reset_index(drop=True)
df_compare_optuna.round(4)

,Representación,Modelo,Validation,F1,Accuracy,Precision,Recall,MCC
0,BoW,XGBoost,Todos,0.5970,0.9330,0.5882,0.6061,0.5606
1,BoW,XGBoost,Sin marcadores,0.6042,0.9137,0.5988,0.6098,0.5558
2,TF-IDF,LinearSVC,Sin marcadores,0.5827,0.9236,0.7105,0.4939,0.5530
3,TF-IDF,LinearSVC,Todos,0.5745,0.9404,0.6923,0.4909,0.5527
4,TF-IDF,XGBoost,Todos,0.5755,0.9261,0.5430,0.6121,0.5363
5,TF-IDF,XGBoost,Sin marcadores,0.5821,0.9045,0.5519,0.6159,0.5294
6,BoW,LinearSVC,Todos,0.5634,0.9231,0.5263,0.6061,0.5230
7,BoW,LinearSVC,Sin marcadores,0.5706,0.9018,0.5410,0.6037,0.5164


### Resultados en test del mejor modelo

Elegimos el candidato inicial o ajustado con mayor MCC en validation completo.
Validation sin marcadores sirve para comparar su comportamiento en ese subconjunto.
El ganador se evalúa en todo test, sin reentrenar ni ajustar nada con test.

In [23]:
# Elegir el ganador antes de cargar test (todos se entrenaron con todo train).
candidates = pd.concat([
    df_compare.assign(Etapa="Inicial"),
    df_compare_optuna.assign(Etapa="Optuna"),
], ignore_index=True)
best = candidates[candidates["Validation"] == "Todos"].sort_values(
    ["MCC", "F1", "Accuracy"], ascending=False, kind="stable"
).iloc[0]
print("Modelo elegido en validation:")
print(best)
pipe = trained_models[(best["Representación"], best["Modelo"], best["Etapa"])]

# Test (se usa solo para la evaluación final)
test = pd.read_json("../data/interim/test_tok.jsonl", lines=True)
prefix = {"TF-IDF": "tfidf", "BoW": "bow"}[best["Representación"]]
X_test = load_npz(f"../data/processed/{prefix}_test.npz")
y_test = test["jailbreak"]
assert X_test.shape[0] == len(y_test)
assert X_test.shape[1] == pipe.n_features_in_

y_pred = pipe.predict(X_test)

df_test_compare = pd.DataFrame([{
    "Representación": best["Representación"],
    "Modelo": best["Modelo"],
    "Etapa": best["Etapa"],
    **evaluate(y_test, y_pred),
}])
df_test_compare.round(4)

Modelo elegido en validation:
Representación         BoW
Modelo             XGBoost
Validation           Todos
F1                0.597015
Accuracy          0.933002
Precision         0.588235
Recall            0.606061
MCC               0.560562
Etapa               Optuna
Name: 8, dtype: object


,Representación,Modelo,Etapa,F1,Accuracy,Precision,Recall,MCC
0,BoW,XGBoost,Optuna,0.648,0.9376,0.6629,0.6339,0.614
